# 작업형 제3유형 — 기본 요점정리

**생성일**: 2026-05-05  
**목적**: ADP 시험 작업형 제3유형(통계·모델링) 핵심 개념 정리  

---

## 목차
- [A. 로지스틱 회귀 (Logistic Regression)](#A)
- [B. 선형 회귀 (Linear Regression)](#B)
- [C. 가설 검정 (Hypothesis Testing)](#C)
- [D. 분산 분석 (ANOVA)](#D)
- [E. 카이제곱 검정 (Chi-Square Test)](#E)
- [F. 생존 분석 (Survival Analysis)](#F)
- [G. 기타 통계 기법](#G)
- [오답 보강 섹션](#error-reinforcement)

---
## A. 로지스틱 회귀 (Logistic Regression) <a id='A'></a>

### 개념 요약
- **목적**: 이진(또는 다항) 종속 변수 예측
- **라이브러리**: `statsmodels.formula.api` (ADP 시험 표준)
- **핵심 함수**: `smf.logit(formula, data).fit(maxiter=200)`
- **결과 해석**:
  - `model.pvalues` : 각 변수의 유의확률
  - `model.params`  : 회귀계수 (로그-오즈)
  - `np.exp(coef)`  : 오즈비 (Odds Ratio)
  - `model.predict(new_data)` : 예측 확률

### 범주형 변수 처리 원칙
| 변수 유형 | 처리 방법 | 예시 |
|-----------|-----------|------|
| 명목형 (Nominal) | `C(변수명)` — 더미 코딩 | 성별, 지역 |
| 순서형 (Ordinal) | 수치형 그대로 투입 | overtime(0/1/2), 학력수준 |
| 이진형 (Binary) | 수치형 (0/1) 그대로 투입 | 흡연여부 |

### 수렴 설정
- 기본 `maxiter=35` → 복잡한 모델에서 수렴 실패 가능
- 해결책: `.fit(maxiter=200)` 또는 `.fit(method='bfgs', maxiter=500)`
- 수렴 경고(`Maximum number of iterations has been exceeded`) 발생 시 계수값 불신뢰

In [ ]:
# [A] 로지스틱 회귀 — 기본 패턴 (ADP 시험 표준)
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

# 1. 모델 적합
model = smf.logit("target ~ var1 + var2 + var3", data=df).fit(maxiter=200)
print(model.summary())

# 2. 유의 변수 회귀계수 (절편 제외)
pvalues = model.pvalues
params = model.params
sig_mask = pvalues.drop('Intercept') < 0.05
sig_coef = params.drop('Intercept')[sig_mask]
print(np.round(sig_coef, 3))

# 3. 오즈비
odds_ratio = np.exp(model.params['var1'])
print(round(odds_ratio, 3))

# 4. 예측 확률
test_data = pd.DataFrame({"var1": [20], "var2": [3000], "var3": [2]})
pred_prob = model.predict(test_data)
print(round(pred_prob[0], 3))

---
## B. 선형 회귀 (Linear Regression) <a id='B'></a>

### 개념 요약
- **목적**: 연속형 종속 변수 예측
- **핵심 함수**: `smf.ols(formula, data).fit()`
- **결과 해석**:
  - `model.rsquared` : 결정계수 R²
  - `model.rsquared_adj` : 수정 결정계수
  - `model.pvalues` : 유의확률
  - `model.params` : 회귀계수

In [ ]:
# [B] 선형 회귀 — 기본 패턴
import statsmodels.formula.api as smf
import numpy as np

model = smf.ols("target ~ var1 + var2 + C(cat_var)", data=df).fit()
print(model.summary())

# 유의 변수 (절편 제외)
pvalues = model.pvalues
params = model.params
sig_coef = params.drop('Intercept')[pvalues.drop('Intercept') < 0.05]
print(np.round(sig_coef, 3))

# R² 출력
print(round(model.rsquared_adj, 3))

---
## C. 가설 검정 (Hypothesis Testing) <a id='C'></a>

### 주요 검정 종류
| 검정 | 함수 | 귀무가설 |
|------|------|----------|
| 단일 표본 t검정 | `scipy.stats.ttest_1samp` | 모평균 = 특정값 |
| 독립 표본 t검정 | `scipy.stats.ttest_ind` | 두 집단 평균 동일 |
| 대응 표본 t검정 | `scipy.stats.ttest_rel` | 차이 평균 = 0 |
| 정규성 검정 | `scipy.stats.shapiro` | 정규 분포를 따름 |
| 등분산 검정 | `scipy.stats.levene` | 분산 동일 |

In [ ]:
# [C] 가설 검정 — 기본 패턴
from scipy import stats

# 독립 표본 t검정
group1 = df[df['group'] == 'A']['value']
group2 = df[df['group'] == 'B']['value']
t_stat, p_value = stats.ttest_ind(group1, group2)
print(round(t_stat, 3), round(p_value, 3))

# 판정: p < 0.05 이면 귀무가설 기각
if p_value < 0.05:
    print("귀무가설 기각: 두 집단 평균 차이 유의")
else:
    print("귀무가설 채택: 두 집단 평균 차이 없음")

---
## D. 분산 분석 (ANOVA) <a id='D'></a>

### 개념 요약
- **목적**: 3개 이상 집단 평균 비교
- **일원 배치**: `scipy.stats.f_oneway`
- **사후 검정**: `statsmodels.stats.multicomp.pairwise_tukeyhsd`
- **귀무가설**: 모든 집단의 평균이 동일

In [ ]:
# [D] 분산 분석 — 기본 패턴
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# 일원 배치 ANOVA
groups = [df[df['group'] == g]['value'] for g in df['group'].unique()]
f_stat, p_value = stats.f_oneway(*groups)
print(round(f_stat, 3), round(p_value, 3))

# 사후 검정 (Tukey HSD)
result = pairwise_tukeyhsd(df['value'], df['group'])
print(result)

---
## E. 카이제곱 검정 (Chi-Square Test) <a id='E'></a>

### 개념 요약
- **목적**: 범주형 변수 간 독립성 검정
- **함수**: `scipy.stats.chi2_contingency`
- **귀무가설**: 두 변수는 독립적이다 (연관성 없음)

In [ ]:
# [E] 카이제곱 검정 — 기본 패턴
from scipy import stats
import pandas as pd

# 분할표 생성
ct = pd.crosstab(df['var1'], df['var2'])
chi2, p_value, dof, expected = stats.chi2_contingency(ct)
print(round(chi2, 3), round(p_value, 3), dof)

---
## F. 생존 분석 (Survival Analysis) <a id='F'></a>

### 개념 요약
- **목적**: 사건 발생까지의 시간 분석
- **라이브러리**: `lifelines`
- **주요 모델**: `KaplanMeierFitter`, `CoxPHFitter`
- **핵심 개념**: 중도절단(censoring), 생존함수, 위험함수

In [ ]:
# [F] 생존 분석 — 기본 패턴
from lifelines import KaplanMeierFitter, CoxPHFitter

# Kaplan-Meier
kmf = KaplanMeierFitter()
kmf.fit(df['duration'], event_observed=df['event'])
print(kmf.median_survival_time_)

# Cox 비례 위험 모형
cph = CoxPHFitter()
cph.fit(df, duration_col='duration', event_col='event')
cph.print_summary()

---
## G. 기타 통계 기법 <a id='G'></a>

### 포함 항목
- **주성분 분석 (PCA)**: `sklearn.decomposition.PCA`
- **군집 분석 (Clustering)**: `sklearn.cluster.KMeans`
- **상관 분석**: `df.corr()`, `scipy.stats.pearsonr`
- **비모수 검정**: `scipy.stats.mannwhitneyu`, `scipy.stats.wilcoxon`

In [ ]:
# [G] 상관 분석 — 기본 패턴
from scipy import stats

r, p = stats.pearsonr(df['var1'], df['var2'])
print(round(r, 3), round(p, 3))

# 비모수: Mann-Whitney U
u_stat, p_value = stats.mannwhitneyu(group1, group2, alternative='two-sided')
print(round(u_stat, 3), round(p_value, 3))

---
## 오답 보강 섹션 <a id='error-reinforcement'></a>

이 섹션은 오답 분석 에이전트로부터 라우팅된 항목을 보강합니다.  
각 보강 항목은 `### ⚠️ 오답 보강` 헤더로 시작합니다.

### ⚠️ 오답 보강 — 유의성 검정은 statsmodels OLS, sklearn LinearRegression은 불가 [9회 Q3-1]

- **발생 오류 유형**: TOOL_ERROR
- **틀린 이유**: `sklearn.linear_model.LinearRegression`은 회귀계수(coef_)만 반환하고 p-value를 제공하지 않는다. "유의하지 않은 설명변수 개수"를 구하려면 반드시 `statsmodels.formula.api.smf.ols()` 를 사용해 `.pvalues`를 확인해야 한다.

| 라이브러리 | 회귀계수 | p-value | 용도 |
|---|---|---|---|
| `sklearn LinearRegression` | ✓ | ✗ | 예측(predict), RMSE 계산 |
| `statsmodels smf.ols` | ✓ | ✓ | 유의성 검정, 회귀 분석 결과 해석 |

**시험 전략**: 유의변수 확인 → `smf.ols().fit()` → 유의하지 않은 변수 제거 → `sklearn LinearRegression`으로 재적합 → RMSE/피어슨 계산


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

# 9회 Q3-1 전체 풀이 패턴 (다중회귀 유의성 → 재적합 → 피어슨/RMSE)

df = pd.read_csv("https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_3_1.csv")
df_train = df.iloc[:140, :]
df_test  = df.iloc[140:, :]

# ── STEP 1: statsmodels OLS로 유의성 검정 ──────────────────────────────
formula = "design ~ tenure + f2 + f3 + f4 + f5"  # id는 불필요 컬럼 제외
ols_model = smf.ols(formula, data=df_train).fit()

# 유의하지 않은 변수 확인 (p-value >= 0.05, 절편 포함)
pvalues = ols_model.pvalues
insig = pvalues[pvalues >= 0.05]
print("유의하지 않은 변수 개수:", len(insig))
print(insig)

# ── STEP 2: 유의하지 않은 변수 제거 후 sklearn으로 재적합 ──────────────
# 제거 변수: f3, f5 (+ id는 원래 제외)
X_train = df_train.drop(["id", "design", "f3", "f5"], axis=1)
y_train = df_train["design"]

# ❌ 틀린 코드 — X_test에 design 미제거 → 피처 수 불일치 또는 data leakage
# X_test = df_test.drop(["id", "f3", "f5"], axis=1)

# ✅ 올바른 코드 — X_test에서도 타겟(design) 반드시 제거
X_test = df_test.drop(["id", "design", "f3", "f5"], axis=1)
y_test = df_test["design"]

model = LinearRegression()
model.fit(X_train, y_train)

# ── STEP 3: 훈련 데이터 피어슨 상관계수 ──────────────────────────────
# ❌ 틀린 코드 — 테스트 데이터 사용 (문제 조건 위반)
# y_test_pred = model.predict(X_test)
# pearson_corr, _ = pearsonr(y_test, y_test_pred)

# ✅ 올바른 코드 — 훈련 데이터 예측값 vs 실제값
y_train_pred = model.predict(X_train)
pearson_corr, _ = pearsonr(y_train, y_train_pred)
print("훈련 데이터 피어슨 상관계수:", round(pearson_corr, 3))

# ── STEP 4: 테스트 데이터 RMSE ────────────────────────────────────────
y_test_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
print("테스트 RMSE:", round(rmse, 3))


### ⚠️ 오답 보강 — 로지스틱 회귀 수렴 설정 및 범주형 변수 처리 (C() vs 수치형)

- **발생 오류 유형**: UNKNOWN (# 모르는 것 주석)
- **문제**: `C(overtime)`으로 범주형 처리 시 더미 변수 수가 늘어나 수렴 실패 가능성. `overtime`이 순서형(0<1<2)이면 수치형 처리가 올바름.
- **수렴 경고 의미**: `"Maximum number of iterations has been exceeded"` → 계수값 불신뢰. `maxiter=200` 등으로 증가 필요.

In [ ]:
# ❌ 틀린 코드 — C(overtime) 범주형 처리 + 수렴 경고 발생
import statsmodels.formula.api as smf
model = smf.logit("attrition ~ age + income + C(overtime)", data=df).fit()
# Warning: Maximum number of iterations has been exceeded

In [ ]:
# ✅ 올바른 코드 — overtime 수치형, maxiter 증가, 절편 제외
import statsmodels.formula.api as smf
import numpy as np

# overtime이 순서형(0,1,2)이면 수치형으로 처리 (C() 불필요)
model = smf.logit("attrition ~ age + income + overtime", data=df).fit(maxiter=200)

pvalues = model.pvalues
params = model.params

# 절편(Intercept) 제외 + 유의확률 0.05 미만 필터링
sig_vars = params.drop('Intercept')[pvalues.drop('Intercept') < 0.05]
print(np.round(sig_vars, 3))

In [ ]:
# ✅ 로지스틱 회귀 전체 패턴 (ADP 시험 표준)
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

# 1. 모델 적합 (수치형 변수 직접 사용, maxiter 충분히)
model = smf.logit("target ~ var1 + var2 + var3", data=df).fit(maxiter=200)

pvalues = model.pvalues
params = model.params

# 2. 유의 변수 회귀계수 (절편 제외)
sig_mask = pvalues.drop('Intercept') < 0.05
sig_coef = params.drop('Intercept')[sig_mask]
print(np.round(sig_coef, 3))

# 3. 오즈비 (특정 변수)
odds_ratio = np.exp(model.params['var1'])
print(round(odds_ratio, 3))

# 4. 예측 확률
test_data = pd.DataFrame({"var1": [20], "var2": [3000], "var3": [2]})
pred_prob = model.predict(test_data)
print(round(pred_prob[0], 3))

> 💡 시험 팁 1: ADP 로지스틱 회귀에서 범주형 처리 `C()`는 명확한 명목형 변수(ex: 성별, 지역)에만 사용. 0/1/2처럼 순서가 있는 변수는 수치형 그대로 투입할 것.
> 
> 💡 시험 팁 2: 수렴 경고가 뜨면 반드시 `.fit(maxiter=200)` 또는 `.fit(method='bfgs', maxiter=500)`으로 수렴 보장 필요. 경고 상태의 계수는 채점 불이익 가능.